# Mini-Projeto 1 — SBC para Recomendação de Cultivo de Plantas

Sistema Baseado em Conhecimento utilizando regras IF-THEN e Experta.

Aluno: José Artur Soares Afreu

Disciplina: Sistemas Baseados em Conhecimento

Prof. Daniel Faustino Lacerda de Souza

## Apresentação do Projeto

Este sistema especialista atua como um assistente inteligente para a recomendação de cultivo de plantas. O sistema recebe características biológicas e ambientais de uma planta e utiliza regras de produção lógicas (*If-Then*) para inferir o manejo adequado e as melhores condições de plantio.

### Objetivos
O objetivo principal é demonstrar a aplicação prática de Sistemas Baseados em Conhecimento (SBC) na resolução de problemas de classificação e recomendação, mapeando o conhecimento agronômico em um motor de inferência. O diagnóstico final unifica decisões sobre:
- Necessidade de luz e alocação ideal no ambiente;
- Perfil de drenagem e receita de substrato;
- Plano de rega;
- Tamanho do vaso;
- Manejo nutricional específico.

### Metodologia e Ferramentas
O projeto foi desenvolvido utilizando a biblioteca `experta`, uma ferramenta em Python focada na criação de sistemas especialistas baseados no paradigma do CLIPS. O motor de inferência opera através de **encadeamento para frente (Forward Chaining)**: parte-se dos fatos iniciais declarados (características da planta) e o motor avalia e dispara as regras aplicáveis de forma cíclica até atingir uma recomendação consolidada (Decisão Final).

## Análise da Lógica de Salience e Resolução de Conflitos

No desenvolvimento de Sistemas Baseados em Conhecimento (SBC) com encadeamento para frente, é comum que múltiplos fatos ativem as condições (*IF*) de diferentes regras simultaneamente. Quando isso ocorre, essas regras entram em uma estrutura lógica chamada **Conjunto de Conflito** (*Conflict Set*).

Para determinar a ordem de execução dessas regras concorrentes, utilizamos a propriedade de **Salience** (prioridade). No motor de inferência deste projeto (`OraculoVaranda`), a prioridade padrão de uma regra é `0` se não declarada, e valores maiores de *salience* forçam a regra a subir na pilha de execução do motor.

### Cenários Práticos de Resolução de Conflitos no Projeto

Podemos observar o funcionamento do *Salience* em dois cenários fundamentais mapeados em nossos testes:

#### 1. Priorização de Nutrição Específica (Adubação de Frutos/Flores)
* **O Conflito:** No caso do teste da **Pimenteira**, a planta possui a propriedade `producao="fruto"`. Isso ativa simultaneamente:
  * A regra geral **`r18_nutri_geral`** (NPK 10-10-10 para manutenção).
  * A regra específica **`r16_nutri_fruto`** (NPK 04-14-08 para frutificação).
* **A Resolução:** Definimos o *salience* de `r16_nutri_fruto` como `320` e o de `r18_nutri_geral` como `300`. Graças ao *salience* superior, a regra de frutificação é disparada primeiro, declarando o fato `NecessidadeNutricional`. Quando a regra geral tenta executar, a validação `NOT(NecessidadeNutricional())` impede que ela sobrescreva a decisão já tomada.

#### 2. Tratamento de Exceções Críticas (Cultivo Aquático)
* **O Conflito:** No teste da **Jiboia na Água**, a planta possui a propriedade `cultivo="agua"`. Variáveis como drenagem, solo e umidade de terra perdem o sentido e poderiam gerar conflitos se processadas pelo motor.
* **A Resolução:** A regra **`r3_agua`** foi configurada com o *salience* máximo do sistema (**`salience=500`**). Isso garante que, no momento em que o motor inicia, a exceção de cultivo hidropônico/aquático passe na frente de qualquer tradução biológica de terra. Ela é executada imediatamente, isola o diagnóstico e permite que a decisão final (`r22_decisao_final`) seja consolidada sem interferência de regras de solo.

In [10]:
# Instalação das dependências do motor de inferência
!pip install experta frozendict==1.2 -q

In [11]:
# Ajuste de compatibilidade da biblioteca com versões recentes do Python
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping

In [12]:
# Importação das dependências e construção do Sistema Baseado em Conhecimento
from experta import KnowledgeEngine, Rule, Fact, MATCH, AS, AND, OR, NOT
import logging

# 1. DEFINIÇÃO DOS FATOS
class Planta(Fact): pass
class PerfilDrenagem(Fact): pass
class ReceitaSubstrato(Fact): pass
class PlanoRega(Fact): pass
class LocalRecomendado(Fact): pass
class VasoRecomendado(Fact): pass
class NecessidadeNutricional(Fact): pass
class DecisaoFinal(Fact): pass

# 2. MOTOR DE INFERÊNCIA
class OraculoVaranda(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.trace = []

    # =========================================================================
    # NÍVEL 1: EXCEÇÕES (Sementes e Água) - Pulam direto para a receita
    # =========================================================================

    @Rule(Planta(fase="semente", plantio="estufa", nome=MATCH.n))
    def r1_semente_estufa(self, n):
        self.trace.append("R1 disparou: Semente sensível. Configurando estufa no pote.")
        self.declare(LocalRecomendado(local="Janela fumê (Estufa improvisada)"))
        self.declare(ReceitaSubstrato(base="Papel toalha úmido ou substrato puro"))
        self.declare(PlanoRega(frequencia="Manter pote fechado para segurar umidade"))
        self.declare(VasoRecomendado(tamanho="Pote transparente fechado"))
        self.declare(NecessidadeNutricional(nivel="zero", formulacao="Nenhuma (Aguardar brotar)"))

    @Rule(Planta(fase="semente", plantio="direto", nome=MATCH.n))
    def r2_semente_direto(self, n):
        self.trace.append("R2 disparou: Semente rústica. Configurando plantio direto.")
        self.declare(LocalRecomendado(local="Sombra da varanda com luz indireta"))
        self.declare(ReceitaSubstrato(base="Substrato da loja puro"))
        self.declare(PlanoRega(frequencia="Manter solo sempre úmido"))
        self.declare(VasoRecomendado(tamanho="Vaso definitivo pequeno"))
        self.declare(NecessidadeNutricional(nivel="zero", formulacao="Nenhuma (Aguardar enraizar)"))

    @Rule(Planta(cultivo="agua", nome=MATCH.n), salience=500)
    def r3_agua(self, n):
        self.trace.append("R3 disparou (salience=500): Cultivo aquático anula regras de terra.")
        self.declare(LocalRecomendado(local="Sombra da varanda"))
        self.declare(ReceitaSubstrato(base="Apenas água"))
        self.declare(PlanoRega(frequencia="Trocar a água semanalmente"))
        self.declare(VasoRecomendado(tamanho="Recipiente de vidro"))
        self.declare(NecessidadeNutricional(nivel="baixa", formulacao="Gotas de NPK líquido (opcional)"))

    # =========================================================================
    # NÍVEL 1: TRADUÇÃO BIOLÓGICA (Plantas na Terra)
    # =========================================================================

    # Drenagem -> Perfil de Solo
    @Rule(Planta(drenagem="alta"))
    def r4_drenagem_alta(self):
        self.trace.append("R4 disparou: Drenagem alta exige perfil drenante.")
        self.declare(PerfilDrenagem(tipo="drenante"))

    @Rule(Planta(drenagem="media"))
    def r5_drenagem_media(self):
        self.trace.append("R5 disparou: Drenagem média exige perfil equilibrado.")
        self.declare(PerfilDrenagem(tipo="equilibrado"))

    @Rule(Planta(drenagem="baixa"))
    def r6_drenagem_baixa(self):
        self.trace.append("R6 disparou: Drenagem baixa exige perfil retentivo.")
        self.declare(PerfilDrenagem(tipo="retentivo"))

    # Umidade -> Rega
    @Rule(Planta(umidade="alta"))
    def r7_rega_alta(self):
        self.trace.append("R7 disparou: Alta umidade demanda rega frequente.")
        self.declare(PlanoRega(frequencia="Frequente"))

    @Rule(Planta(umidade="media"))
    def r8_rega_media(self):
        self.trace.append("R8 disparou: Umidade média demanda rega moderada.")
        self.declare(PlanoRega(frequencia="Moderada"))

    @Rule(Planta(umidade="baixa"))
    def r9_rega_baixa(self):
        self.trace.append("R9 disparou: Baixa umidade permite rega espaçada.")
        self.declare(PlanoRega(frequencia="Espaçada"))

    # Luz -> Local (Bloqueado caso a planta seja aquática, para não gerar locais duplicados)
    @Rule(Planta(luz="sol_pleno"), NOT(Planta(cultivo="agua")))
    def r10_luz_sol(self):
        self.trace.append("R10 disparou: Sol pleno aloca na Cima da Varanda (Norte).")
        self.declare(LocalRecomendado(local="Cima da varanda (Abertura Norte)"))

    @Rule(Planta(luz="meia_sombra"), NOT(Planta(cultivo="agua")))
    def r11_luz_meia(self):
        self.trace.append("R11 disparou: Meia sombra aloca no piso da varanda.")
        self.declare(LocalRecomendado(local="Sombra da varanda (Luz no piso)"))

    @Rule(Planta(luz="sombra"), NOT(Planta(cultivo="agua")))
    def r12_luz_sombra(self):
        self.trace.append("R12 disparou: Sombra aloca no interior da casa.")
        self.declare(LocalRecomendado(local="Janela do quarto (Fumê)"))

    # Tamanho -> Vaso
    @Rule(Planta(tamanho="grande"))
    def r13_vaso_grande(self):
        self.trace.append("R13 disparou: Porte grande exige vaso grande.")
        self.declare(VasoRecomendado(tamanho="Grande"))

    @Rule(Planta(tamanho="medio"))
    def r14_vaso_medio(self):
        self.trace.append("R14 disparou: Porte médio exige vaso médio.")
        self.declare(VasoRecomendado(tamanho="Médio"))

    @Rule(Planta(tamanho="pequeno"))
    def r15_vaso_pequeno(self):
        self.trace.append("R15 disparou: Porte pequeno exige vaso pequeno.")
        self.declare(VasoRecomendado(tamanho="Pequeno"))

    # Nutrição (Estratégia de Conflito com Salience e NOT)
    @Rule(Planta(producao="fruto"), salience=320)
    def r16_nutri_fruto(self):
        self.trace.append("R16 disparou (salience=320): Produção de frutos vence NPK geral.")
        self.declare(NecessidadeNutricional(nivel="alto", formulacao="NPK 04-14-08 (Frutificação)"))

    @Rule(Planta(producao="flor"), salience=315)
    def r17_nutri_flor(self):
        self.trace.append("R17 disparou (salience=315): Produção floral vence NPK geral.")
        self.declare(NecessidadeNutricional(nivel="alto", formulacao="NPK 04-14-08 (Floração)"))

    @Rule(Planta(), NOT(NecessidadeNutricional()), salience=300)
    def r18_nutri_geral(self):
        self.trace.append("R18 disparou (salience=300 via NOT): Ativando NPK de manutenção.")
        self.declare(NecessidadeNutricional(nivel="medio", formulacao="NPK 10-10-10 (Manutenção) + Húmus"))

    # =========================================================================
    # NÍVEL 2: O ALQUIMISTA (Cruzando Perfis em Receitas Físicas)
    # =========================================================================

    @Rule(PerfilDrenagem(tipo="drenante"))
    def r19_rec_drenante(self):
        self.trace.append("R19 disparou: Receita drenante formulada.")
        self.declare(ReceitaSubstrato(base="Areia de construção, Brita no fundo e Substrato da loja"))

    @Rule(PerfilDrenagem(tipo="equilibrado"))
    def r20_rec_equilibrado(self):
        self.trace.append("R20 disparou: Receita equilibrada formulada.")
        self.declare(ReceitaSubstrato(base="Substrato da loja misturado com Húmus de minhoca e pouca Areia"))

    @Rule(PerfilDrenagem(tipo="retentivo"), NOT(PerfilDrenagem(tipo="drenante")))
    def r21_rec_rica(self):
        self.trace.append("R21 disparou: Receita rica formulada com validação NOT.")
        self.declare(ReceitaSubstrato(base="Substrato da Mata Atlântica, Húmus de minhoca e Esterco"))

    # =========================================================================
    # NÍVEL 3: DECISÃO FINAL UNIFICADA
    # =========================================================================

    @Rule(
        ReceitaSubstrato(base=MATCH.sub),
        PlanoRega(frequencia=MATCH.reg),
        LocalRecomendado(local=MATCH.loc),
        VasoRecomendado(tamanho=MATCH.vaso),
        NecessidadeNutricional(formulacao=MATCH.nutri),
        Planta(nome=MATCH.nome_planta),
        NOT(DecisaoFinal()) # Previne loop infinito
    )
    def r22_decisao_final(self, sub, reg, loc, vaso, nutri, nome_planta):
        self.trace.append(f"R22 disparou: Diagnóstico completo consolidado para '{nome_planta}'.")
        self.declare(DecisaoFinal(
            planta=nome_planta, local=loc, substrato=sub, rega=reg, vaso=vaso, nutricao=nutri
        ))

In [13]:
# Função auxiliar para executar e visualizar os testes do motor de inferência
def testar_oraculo(titulo, planta_fact):
    print("=" * 70)
    print(f"CASO: {titulo}")
    print("=" * 70)

    motor = OraculoVaranda()
    motor.reset()
    motor.declare(planta_fact)
    motor.run()

    print("\nTRACE DE RACIOCÍNIO:")
    for passo in motor.trace:
        print(f" -> {passo}")

    print("\nDECISÃO CONSOLIDADA:")
    for f in motor.facts.values():
        if isinstance(f, DecisaoFinal):
            print(f"Planta: {f['planta']}")
            print(f"Local: {f['local']}")
            print(f"Substrato: {f['substrato']}")
            print(f"Rega: {f['rega']}")
            print(f"Vaso: {f['vaso']}")
            print(f"Nutrição: {f['nutricao']}")
    print("\n")

In [14]:
# TESTE 1: Planta Completa (Fruto + Sol Pleno)
testar_oraculo(
    "Pimenteira (Teste de Salience R16 vs R18)",
    Planta(nome="Pimenteira", luz="sol_pleno", drenagem="alta", umidade="media", tamanho="grande", producao="fruto")
)

CASO: Pimenteira (Teste de Salience R16 vs R18)

TRACE DE RACIOCÍNIO:
 -> R16 disparou (salience=320): Produção de frutos vence NPK geral.
 -> R13 disparou: Porte grande exige vaso grande.
 -> R8 disparou: Umidade média demanda rega moderada.
 -> R4 disparou: Drenagem alta exige perfil drenante.
 -> R19 disparou: Receita drenante formulada.
 -> R10 disparou: Sol pleno aloca na Cima da Varanda (Norte).
 -> R22 disparou: Diagnóstico completo consolidado para 'Pimenteira'.

DECISÃO CONSOLIDADA:
Planta: Pimenteira
Local: Cima da varanda (Abertura Norte)
Substrato: Areia de construção, Brita no fundo e Substrato da loja
Rega: Moderada
Vaso: Grande
Nutrição: NPK 04-14-08 (Frutificação)




In [15]:
# TESTE 2: Planta Omitindo Produção (Teste do NOT ativando R18)
testar_oraculo(
    "Zamioculca (Omitindo produção para forçar NPK geral)",
    Planta(nome="Zamioculca", luz="sombra", drenagem="alta", umidade="baixa", tamanho="pequeno")
)

CASO: Zamioculca (Omitindo produção para forçar NPK geral)

TRACE DE RACIOCÍNIO:
 -> R18 disparou (salience=300 via NOT): Ativando NPK de manutenção.
 -> R15 disparou: Porte pequeno exige vaso pequeno.
 -> R4 disparou: Drenagem alta exige perfil drenante.
 -> R19 disparou: Receita drenante formulada.
 -> R9 disparou: Baixa umidade permite rega espaçada.
 -> R12 disparou: Sombra aloca no interior da casa.
 -> R22 disparou: Diagnóstico completo consolidado para 'Zamioculca'.

DECISÃO CONSOLIDADA:
Planta: Zamioculca
Local: Janela do quarto (Fumê)
Substrato: Areia de construção, Brita no fundo e Substrato da loja
Rega: Espaçada
Vaso: Pequeno
Nutrição: NPK 10-10-10 (Manutenção) + Húmus




In [16]:
# TESTE 3: Semente Sensível (Estufa)
testar_oraculo(
    "Semente de Jurema (Semente Estufa)",
    Planta(nome="Semente de Jurema", fase="semente", plantio="estufa")
)

CASO: Semente de Jurema (Semente Estufa)

TRACE DE RACIOCÍNIO:
 -> R18 disparou (salience=300 via NOT): Ativando NPK de manutenção.
 -> R1 disparou: Semente sensível. Configurando estufa no pote.
 -> R22 disparou: Diagnóstico completo consolidado para 'Semente de Jurema'.

DECISÃO CONSOLIDADA:
Planta: Semente de Jurema
Local: Janela fumê (Estufa improvisada)
Substrato: Papel toalha úmido ou substrato puro
Rega: Manter pote fechado para segurar umidade
Vaso: Pote transparente fechado
Nutrição: Nenhuma (Aguardar brotar)




In [17]:
# TESTE 4: Semente Rústica (Plantio Direto)
testar_oraculo(
    "Semente de Tomate (Plantio Direto)",
    Planta(nome="Semente de Tomate Cereja", fase="semente", plantio="direto")
)

CASO: Semente de Tomate (Plantio Direto)

TRACE DE RACIOCÍNIO:
 -> R18 disparou (salience=300 via NOT): Ativando NPK de manutenção.
 -> R2 disparou: Semente rústica. Configurando plantio direto.
 -> R22 disparou: Diagnóstico completo consolidado para 'Semente de Tomate Cereja'.

DECISÃO CONSOLIDADA:
Planta: Semente de Tomate Cereja
Local: Sombra da varanda com luz indireta
Substrato: Substrato da loja puro
Rega: Manter solo sempre úmido
Vaso: Vaso definitivo pequeno
Nutrição: Nenhuma (Aguardar enraizar)




In [18]:
# TESTE 5: Exceção Aquática (Ignorando sol e terra)
testar_oraculo(
    "Muda de Jiboia na Água (Salience 500)",
    Planta(nome="Muda de Jiboia", cultivo="agua", luz="sol_pleno")
)

CASO: Muda de Jiboia na Água (Salience 500)

TRACE DE RACIOCÍNIO:
 -> R3 disparou (salience=500): Cultivo aquático anula regras de terra.
 -> R22 disparou: Diagnóstico completo consolidado para 'Muda de Jiboia'.

DECISÃO CONSOLIDADA:
Planta: Muda de Jiboia
Local: Sombra da varanda
Substrato: Apenas água
Rega: Trocar a água semanalmente
Vaso: Recipiente de vidro
Nutrição: Gotas de NPK líquido (opcional)


